# 🧠 HMS 有害脑活动分类 - Colab训练

**Kaggle Competition**: [HMS - Harmful Brain Activity Classification](https://www.kaggle.com/competitions/hms-harmful-brain-activity-classification)

## 使用说明
1. 修改运行时类型为GPU: `运行时` → `更改运行时类型` → `T4 GPU`
2. 按顺序执行每个cell
3. 首次运行需要配置Kaggle API并下载数据（约30-60分钟）

## 1️⃣ 环境检查与配置

In [ ]:
# 检查GPU
!nvidia-smi

In [ ]:
# 安装依赖
!pip install timm einops pyyaml tqdm kaggle pyarrow --quiet

import torch
print(f"✓ PyTorch版本: {torch.__version__}")
print(f"✓ CUDA可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU型号: {torch.cuda.get_device_name(0)}")
    print(f"✓ GPU显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2️⃣ 配置Kaggle API

1. 访问 https://www.kaggle.com/settings
2. 点击 "Create New Token" 下载 `kaggle.json`
3. 运行下面的cell上传文件

In [ ]:
from google.colab import files
import os

# 检查是否已配置
if os.path.exists('/root/.kaggle/kaggle.json'):
    print("✓ Kaggle API已配置")
else:
    print("请上传kaggle.json文件:")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !mv kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("✓ Kaggle API配置完成!")

## 3️⃣ 挂载Google Drive并下载数据

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 数据目录
DATA_DIR = '/content/drive/MyDrive/HMS_Data'
OUTPUT_DIR = '/content/drive/MyDrive/HMS_Output'

!mkdir -p {DATA_DIR}
!mkdir -p {OUTPUT_DIR}

In [ ]:
import os

# 检查数据是否已下载
if os.path.exists(f'{DATA_DIR}/train.csv'):
    print("✓ 数据已存在，跳过下载")
else:
    print("开始下载数据（约50GB，请耐心等待）...")
    !kaggle competitions download -c hms-harmful-brain-activity-classification -p {DATA_DIR}
    print("解压数据...")
    !cd {DATA_DIR} && unzip -q '*.zip'
    print("✓ 数据下载并解压完成!")

# 显示数据目录内容
!ls -la {DATA_DIR}/

## 4️⃣ 导入库和配置

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from scipy.ndimage import zoom
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True

set_seed(42)

In [ ]:
# 配置类
class Config:
    # 数据路径
    data_dir = '/content/drive/MyDrive/HMS_Data'
    train_csv = f'{data_dir}/train.csv'
    eeg_dir = f'{data_dir}/train_eegs'
    spec_dir = f'{data_dir}/train_spectrograms'
    output_dir = '/content/drive/MyDrive/HMS_Output'

    # 模型参数
    num_classes = 6
    hidden_dim = 128  # 可以调大，但注意显存

    # 训练参数
    batch_size = 16      # T4 GPU建议16，A100可以用32-64
    epochs = 15
    lr = 1e-4
    weight_decay = 0.01
    num_workers = 2
    use_amp = True       # 混合精度训练

    # 数据划分
    fold = 0
    n_folds = 5

    # 设备
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

config = Config()
print(f"Device: {config.device}")
print(f"Batch Size: {config.batch_size}")
print(f"Epochs: {config.epochs}")

## 5️⃣ 数据集定义

In [ ]:
class HMSDataset(Dataset):
    """HMS数据集"""

    def __init__(self, df, eeg_dir, spec_dir, mode='train'):
        self.df = df.reset_index(drop=True)
        self.eeg_dir = eeg_dir
        self.spec_dir = spec_dir
        self.mode = mode
        self.label_cols = ['seizure_vote', 'lpd_vote', 'gpd_vote',
                          'lrda_vote', 'grda_vote', 'other_vote']

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # 加载EEG
        eeg_path = os.path.join(self.eeg_dir, f"{int(row['eeg_id'])}.parquet")
        try:
            eeg_df = pd.read_parquet(eeg_path)
            eeg = eeg_df.iloc[:2000, :20].values.T  # (20, 2000)
            eeg = np.nan_to_num(eeg, nan=0.0).astype(np.float32)
            # 标准化
            mean, std = eeg.mean(), eeg.std()
            eeg = (eeg - mean) / (std + 1e-8)
        except Exception as e:
            eeg = np.zeros((20, 2000), dtype=np.float32)

        # 加载频谱图
        spec_path = os.path.join(self.spec_dir, f"{int(row['spectrogram_id'])}.parquet")
        try:
            spec_df = pd.read_parquet(spec_path)
            spec = spec_df.iloc[:, 1:].values  # 去掉时间列
            spec = np.nan_to_num(spec, nan=0.0).astype(np.float32)
            # 调整大小
            h, w = spec.shape
            spec = zoom(spec, (128/h, 256/w), order=1)
            # 标准化
            mean, std = spec.mean(), spec.std()
            spec = (spec - mean) / (std + 1e-8)
            spec = np.stack([spec] * 4, axis=0)  # (4, 128, 256)
        except Exception as e:
            spec = np.zeros((4, 128, 256), dtype=np.float32)

        # 标签
        votes = row[self.label_cols].values.astype(np.float32)
        label = votes / (votes.sum() + 1e-8)

        return {
            'eeg': torch.tensor(eeg),
            'spec': torch.tensor(spec),
            'label': torch.tensor(label)
        }

## 6️⃣ 模型定义

In [ ]:
class EEGEncoder(nn.Module):
    """EEG编码器 - 1D CNN + LSTM"""
    def __init__(self, in_channels=20, hidden_dim=128, out_dim=128):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, hidden_dim, kernel_size=15, padding=7),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=15, padding=7),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.MaxPool1d(4),  # 2000 -> 500
        )

        self.lstm = nn.LSTM(hidden_dim, hidden_dim//2, bidirectional=True,
                           batch_first=True, num_layers=2, dropout=0.2)

        self.fc = nn.Linear(hidden_dim, out_dim)
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, x):
        x = self.conv(x)  # (B, hidden_dim, 500)
        x = x.permute(0, 2, 1)  # (B, 500, hidden_dim)
        x, _ = self.lstm(x)  # (B, 500, hidden_dim)
        x = x.mean(dim=1)  # (B, hidden_dim)
        x = self.fc(x)
        x = self.norm(x)
        return x


class SpecEncoder(nn.Module):
    """频谱图编码器 - 2D CNN"""
    def __init__(self, in_channels=4, hidden_dim=128, out_dim=128):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(in_channels, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.GELU(),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.MaxPool2d(2),

            # Block 3
            nn.Conv2d(64, hidden_dim, 3, padding=1),
            nn.BatchNorm2d(hidden_dim),
            nn.GELU(),
            nn.MaxPool2d(2),

            # Block 4
            nn.Conv2d(hidden_dim, hidden_dim, 3, padding=1),
            nn.BatchNorm2d(hidden_dim),
            nn.GELU(),
            nn.AdaptiveAvgPool2d(1)
        )

        self.fc = nn.Linear(hidden_dim, out_dim)
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        x = self.fc(x)
        x = self.norm(x)
        return x


class HMSModel(nn.Module):
    """HMS多模态融合模型"""
    def __init__(self, num_classes=6, hidden_dim=128):
        super().__init__()

        self.eeg_encoder = EEGEncoder(20, hidden_dim, hidden_dim)
        self.spec_encoder = SpecEncoder(4, hidden_dim, hidden_dim)

        # 门控融合
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2),
            nn.Softmax(dim=-1)
        )

        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.3)
        )

        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, eeg, spec):
        eeg_feat = self.eeg_encoder(eeg)  # (B, hidden_dim)
        spec_feat = self.spec_encoder(spec)  # (B, hidden_dim)

        # 门控权重
        concat = torch.cat([eeg_feat, spec_feat], dim=-1)
        weights = self.gate(concat)  # (B, 2)

        # 加权融合
        fused = weights[:, 0:1] * eeg_feat + weights[:, 1:2] * spec_feat
        fused = self.fusion(fused)

        logits = self.classifier(fused)
        probs = F.softmax(logits, dim=-1)

        return {'probs': probs, 'logits': logits, 'weights': weights}

## 7️⃣ 训练函数

In [ ]:
def train_epoch(model, loader, optimizer, scaler, device, use_amp=True):
    model.train()
    total_loss = 0

    pbar = tqdm(loader, desc='Training')
    for batch in pbar:
        eeg = batch['eeg'].to(device)
        spec = batch['spec'].to(device)
        label = batch['label'].to(device)

        optimizer.zero_grad()

        with autocast(enabled=use_amp):
            output = model(eeg, spec)
            loss = F.kl_div(output['probs'].log(), label, reduction='batchmean')

        if use_amp:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []

    for batch in tqdm(loader, desc='Validation'):
        eeg = batch['eeg'].to(device)
        spec = batch['spec'].to(device)
        label = batch['label'].to(device)

        output = model(eeg, spec)
        all_preds.append(output['probs'].cpu())
        all_labels.append(label.cpu())

    all_preds = torch.cat(all_preds, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    kl = F.kl_div(all_preds.log(), all_labels, reduction='batchmean').item()

    return kl, all_preds.numpy(), all_labels.numpy()

## 8️⃣ 加载数据

In [ ]:
# 加载数据
print("Loading data...")
df = pd.read_csv(config.train_csv)
print(f"Total samples: {len(df)}")

# 数据划分
gkf = GroupKFold(n_splits=config.n_folds)
patient_ids = df['patient_id'].values

for i, (train_idx, val_idx) in enumerate(gkf.split(df, groups=patient_ids)):
    if i == config.fold:
        train_df = df.iloc[train_idx]
        val_df = df.iloc[val_idx]
        break

print(f"Fold {config.fold}: Train={len(train_df)}, Val={len(val_df)}")

# 创建数据加载器
train_dataset = HMSDataset(train_df, config.eeg_dir, config.spec_dir, 'train')
val_dataset = HMSDataset(val_df, config.eeg_dir, config.spec_dir, 'val')

train_loader = DataLoader(train_dataset, batch_size=config.batch_size,
                         shuffle=True, num_workers=config.num_workers,
                         pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size,
                       shuffle=False, num_workers=config.num_workers,
                       pin_memory=True)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

## 9️⃣ 开始训练

In [ ]:
# 创建模型
model = HMSModel(config.num_classes, config.hidden_dim)
model = model.to(config.device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

# 优化器和调度器
optimizer = optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs)
scaler = GradScaler(enabled=config.use_amp)

In [ ]:
# 训练循环
history = {'train_loss': [], 'val_kl': [], 'lr': []}
best_kl = float('inf')

print("="*60)
print("Starting Training")
print("="*60)

for epoch in range(config.epochs):
    print(f"\n--- Epoch {epoch+1}/{config.epochs} ---")

    # 训练
    train_loss = train_epoch(model, train_loader, optimizer, scaler,
                            config.device, config.use_amp)

    # 验证
    val_kl, preds, labels = validate(model, val_loader, config.device)

    # 调度器
    scheduler.step()
    lr = optimizer.param_groups[0]['lr']

    # 记录
    history['train_loss'].append(train_loss)
    history['val_kl'].append(val_kl)
    history['lr'].append(lr)

    print(f"Train Loss: {train_loss:.4f} | Val KL: {val_kl:.4f} | LR: {lr:.2e}")

    # 保存最佳模型
    if val_kl < best_kl:
        best_kl = val_kl
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_kl': val_kl
        }, f'{config.output_dir}/best_model.pth')
        print(f"  ✓ Saved best model! KL: {val_kl:.4f}")

    # 每5个epoch保存检查点
    if (epoch + 1) % 5 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'history': history
        }, f'{config.output_dir}/checkpoint_epoch{epoch+1}.pth')

print("\n" + "="*60)
print(f"Training Complete! Best Val KL: {best_kl:.4f}")
print("="*60)

## 🔟 可视化结果

In [ ]:
# 绘制训练曲线
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 损失曲线
axes[0].plot(history['train_loss'], 'b-', linewidth=2)
axes[0].set_title('Train Loss', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

# KL散度
axes[1].plot(history['val_kl'], 'r-', linewidth=2)
axes[1].axhline(y=best_kl, color='g', linestyle='--', label=f'Best: {best_kl:.4f}')
best_epoch = np.argmin(history['val_kl'])
axes[1].scatter([best_epoch], [best_kl], color='g', s=100, zorder=5)
axes[1].set_title('Validation KL (Competition Metric)', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('KL Divergence')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 学习率
axes[2].plot(history['lr'], 'g-', linewidth=2)
axes[2].set_title('Learning Rate', fontsize=14)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('LR')
axes[2].set_yscale('log')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{config.output_dir}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFigure saved to: {config.output_dir}/training_curves.png")

In [ ]:
# 保存训练历史
with open(f'{config.output_dir}/history.json', 'w') as f:
    json.dump(history, f, indent=2)

print(f"History saved to: {config.output_dir}/history.json")

## 1️⃣1️⃣ 下载结果

In [ ]:
# 下载模型和结果到本地
from google.colab import files

# 下载最佳模型
files.download(f'{config.output_dir}/best_model.pth')

# 下载训练曲线
files.download(f'{config.output_dir}/training_curves.png')

# 下载训练历史
files.download(f'{config.output_dir}/history.json')

---

## 📝 训练总结

| 指标 | 值 |
|------|----|
| 最佳验证KL | 见上方输出 |
| 最佳Epoch | 见上方输出 |
| 模型参数量 | 见上方输出 |

### 后续步骤
1. 尝试调整超参数（学习率、batch size等）
2. 增加训练轮数
3. 尝试更大的模型（hidden_dim=256）
4. 使用完整版模型（EEGMamba + CMFViT）
5. 集成多个fold的模型